# Agentic Design Patterns for Beginners

## What Are Agentic Design Patterns?

Agentic design patterns are proven approaches for building AI agents that can **reason**, **act**, **learn**, and **make decisions**. Just like design patterns in software engineering, these patterns provide reusable solutions to common problems in agent development.

**Why Patterns Matter:**

- **Proven Solutions** - Battle-tested approaches that work

- **Faster Development** - Don't reinvent the wheel

- **Better Outcomes** - Choose the right pattern for your task

- **Easy to Understand** - Common vocabulary for team communication

---

## What You'll Learn

This notebook covers **7 essential patterns**:

### NEW Implementations (Full working code):

1. **ReAct** - Reasoning + Acting for tool-using agents

2. **Plan-and-Execute** - Two-phase problem solving

3. **Prompt Chaining** - Sequential task decomposition

4. **Routing** - Input classification and specialized handling

### Existing Patterns (Summaries + Links):

5. **Reflection** - Self-critique and improvement

6. **Reflexion** - Learning from past failures

7. **Tool-Calling** - External API integration

**Plus:** References to 5 additional advanced patterns

---

## Prerequisites

- Basic Python knowledge

- Familiarity with LangGraph basics (StateGraph, nodes, edges)

- LLM API key configured

Let's get started! 🚀

In [ ]:
# Setup and imports
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver
from typing import TypedDict, Literal, Annotated
from langchain_core.messages import HumanMessage, AIMessage
from operator import add
import os

load_dotenv()

# Initialize LLM
api_key = os.environ['UNIFIED_LLM_KEY']
base_url = ""

llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0.7,
    max_tokens=300,
    api_key=api_key,
    base_url=base_url
)

print("✅ Setup complete!")

## Pattern Comparison Table


| **Pattern** | **Focus** | **Best For** | **Complexity** |

|-------------|-----------|--------------|----------------|

| **ReAct** | Reasoning + Acting | Tool-using Q&A agents | Medium |

| **Plan-and-Execute** | Two-phase problem solving | Multi-step tasks | Medium |

| **Prompt Chaining** | Sequential decomposition | Processing pipelines | Low |

| **Routing** | Input classification | Diverse input types | Low |

| **Reflection** | Self-improvement | Content generation | Medium |

| **Reflexion** | Learning from failures | Iterative tasks | High |

| **Tool-Calling** | External API integration | Real-time data | Medium |


---

## Quick Decision Guide

**Need to use external tools/APIs?** → ReAct or Tool-Calling

**Complex multi-step task?** → Plan-and-Execute

**Sequential processing?** → Prompt Chaining

**Different input types?** → Routing

**Improve output quality?** → Reflection

**Learn over time?** → Reflexion

---

# 1. ReAct Pattern (Reasoning + Acting)

## What It Does

ReAct combines **reasoning** (thinking through problems) and **acting** (executing actions) in an interleaved manner. The agent alternates between generating thoughts and taking actions to gather information, reducing hallucinations and improving accuracy.

**Flow Diagram:**
```
Question → Thought (Reason) → Action (Tool Use) → Observation →
         ↑_______________________________________________|

```
Repeat until answer is found, then Final Answer

---

## When to Use

✅ **Use ReAct when:**

- Building Q&A agents that need external information

- Agent needs to use tools/APIs to answer questions

- You want visible reasoning steps (explainability)

- Reducing hallucinations is critical

❌ **Don't use when:**

- Simple tasks that don't need external tools

- Real-time performance is critical (adds overhead)

- You don't need explainability

In [ ]:
# ReAct Pattern - State and Nodes

# State schema
class ReActState(TypedDict):
    question: str
    thought: str  # Agent's reasoning
    action: str  # Tool to use or "FINISH"
    observation: str  # Result from tool
    answer: str

# Node 1: Think (reason about what to do)
def think_node(state: ReActState):
    """Agent reasons about the question"""
    prompt = f"""Question: {state['question']}

Think step-by-step about how to answer this question.
If you need information, respond with: ACTION: search
If you can answer, respond with: ACTION: FINISH

Your thought:"""

    response = llm.invoke(prompt)
    thought = response.content

    # Extract action from thought
    if "ACTION: search" in thought or "need to search" in thought.lower():
        action = "search"
    else:
        action = "FINISH"

    print(f"💭 Thought: {thought[:100]}...")
    print(f"🎯 Action: {action}")

    return {"thought": thought, "action": action}

# Node 2: Act (execute tool or finish)
def act_node(state: ReActState):
    """Execute action - simulated search"""
    if state["action"] == "search":
        # Simulate web search (in real implementation, use actual search API)
        observation = f"Search results for '{state['question']}': LangGraph is a framework for building stateful, multi-actor AI agents using LLMs."
        print(f"🔍 Observation: {observation[:80]}...")
        return {"observation": observation}
    else:
        # Generate final answer
        answer = llm.invoke(f"Based on: {state.get('observation', '')}, answer: {state['question']}").content
        print(f"✅ Answer: {answer[:80]}...")
        return {"answer": answer}

In [ ]:
# ReAct Pattern - Graph Construction

# Router function: decide whether to continue or finish
def should_continue(state: ReActState) -> Literal["think", "finish"]:
    """Route based on action"""
    if state["action"] == "FINISH":
        return "finish"
    return "think"  # Loop back to think after observation

# Build graph
react_workflow = StateGraph(ReActState)

# Add nodes
react_workflow.add_node("think", think_node)
react_workflow.add_node("act", act_node)

# Add edges
react_workflow.add_edge(START, "think")
react_workflow.add_edge("think", "act")
react_workflow.add_conditional_edges(
    "act",
    should_continue,
    {
        "think": "think",  # Loop: get more info
        "finish": END      # Done: have answer
    }
)

# Compile
react_agent = react_workflow.compile()

print("✅ ReAct agent built!")

In [ ]:
# ReAct Pattern - Test Example

print("=" * 60)
print("ReAct Pattern Demo")
print("=" * 60)

# Test question
result = react_agent.invoke({
    "question": "What is LangGraph?",
    "thought": "",
    "action": "",
    "observation": "",
    "answer": ""
})

print("\n" + "=" * 60)
print("FINAL RESULT")
print("=" * 60)
print(f"Question: {result['question']}")
print(f"Answer: {result['answer']}")

# 2. Plan-and-Execute Pattern

## What It Does

Plan-and-Execute breaks complex tasks into two phases: **planning** (creating a step-by-step plan) and **execution** (following the plan). This prevents missing steps and maintains structured reasoning for multi-step problems.

In [ ]:
**Flow Diagram:**
Problem → Create Plan → Execute Step 1 → Execute Step 2 → ... → Final Result
          (Planning)    (Execution Phase)

---

## When to Use

✅ **Use Plan-and-Execute when:**

- Complex problems need structured breakdown

- Multiple interdependent steps required

- Want transparency in problem-solving approach

- Mathematical reasoning or multi-step workflows

❌ **Don't use when:**

- Simple single-step tasks

- Plans can't be predetermined (highly dynamic)

- Speed is more important than structure

In [ ]:
# Plan-and-Execute - State and Nodes

# State schema
class PlanExecuteState(TypedDict):
    task: str
    plan: list[str]  # List of steps
    current_step: int
    step_results: Annotated[list[str], add]  # Accumulate results
    final_answer: str

# Node 1: Create Plan
def create_plan(state: PlanExecuteState):
    """Generate step-by-step plan"""
    prompt = f"""Break down this task into clear, numbered steps:

Task: {state['task']}

Provide 3-5 steps as a numbered list."""

    response = llm.invoke(prompt)
    plan_text = response.content

    # Parse into list (simple split by lines starting with numbers)
    steps = [line.strip() for line in plan_text.split('\n') if line.strip() and line[0].isdigit()]

    print(f"📋 Plan created ({len(steps)} steps):")
    for step in steps:
        print(f"  {step}")

    return {"plan": steps, "current_step": 0}

# Node 2: Execute Step
def execute_step(state: PlanExecuteState):
    """Execute current step"""
    current = state["current_step"]
    step = state["plan"][current]

    # Execute step with LLM
    prompt = f"""Execute this step: {step}

Previous results: {state.get('step_results', [])}

Provide the result:"""

    result = llm.invoke(prompt).content

    print(f"✅ Step {current + 1} completed: {result[:60]}...")

    return {
        "step_results": [result],
        "current_step": current + 1
    }

# Node 3: Finalize
def finalize_answer(state: PlanExecuteState):
    """Synthesize final answer"""
    prompt = f"""Synthesize final answer from these step results:

{chr(10).join(f'{i+1}. {r}' for i, r in enumerate(state['step_results']))}

Final answer:"""

    answer = llm.invoke(prompt).content
    print(f"🎯 Final answer generated")
    return {"final_answer": answer}

In [ ]:
# Plan-and-Execute - Graph Construction

# Router: check if more steps remain
def more_steps(state: PlanExecuteState) -> Literal["execute", "finalize"]:
    """Check if there are more steps to execute"""
    if state["current_step"] < len(state["plan"]):
        return "execute"
    return "finalize"

# Build graph
plan_execute_workflow = StateGraph(PlanExecuteState)

# Add nodes
plan_execute_workflow.add_node("plan", create_plan)
plan_execute_workflow.add_node("execute", execute_step)
plan_execute_workflow.add_node("finalize", finalize_answer)

# Add edges
plan_execute_workflow.add_edge(START, "plan")
plan_execute_workflow.add_edge("plan", "execute")
plan_execute_workflow.add_conditional_edges(
    "execute",
    more_steps,
    {
        "execute": "execute",  # Loop: more steps
        "finalize": "finalize"  # Done: finalize
    }
)
plan_execute_workflow.add_edge("finalize", END)

# Compile
plan_execute_agent = plan_execute_workflow.compile()

print("✅ Plan-and-Execute agent built!")

In [ ]:
# Plan-and-Execute - Test Example

print("=" * 60)
print("Plan-and-Execute Pattern Demo")
print("=" * 60)

# Test task
result = plan_execute_agent.invoke({
    "task": "Calculate the total cost of buying 3 notebooks at $5 each and 2 pens at $2 each, then apply a 10% discount",
    "plan": [],
    "current_step": 0,
    "step_results": [],
    "final_answer": ""
})

print("\n" + "=" * 60)
print("FINAL RESULT")
print("=" * 60)
print(f"Task: {result['task']}")
print(f"\nPlan executed:")
for i, step in enumerate(result['plan'], 1):
    print(f"  {i}. {step}")
print(f"\nFinal Answer: {result['final_answer']}")

# 3. Prompt Chaining Pattern

## What It Does

Prompt Chaining decomposes complex tasks into a **sequence of simpler steps**, where each LLM call processes the output of the previous one. Each step focuses on a specific, well-defined subtask, improving accuracy and reducing errors.

In [ ]:
**Flow Diagram:**
Input → Step 1 (Extract) → Step 2 (Transform) → Step 3 (Summarize) → Output
        Each step = focused LLM call

---

## When to Use

✅ **Use Prompt Chaining when:**

- Task can be broken into sequential steps

- Each step needs different prompts or focus

- Want to isolate errors to specific steps

- Improve accuracy over single-shot approaches

❌ **Don't use when:**

- Simple tasks that don't need decomposition

- Steps can't be clearly separated

- Latency is critical (multiple LLM calls add time)

In [ ]:
# Prompt Chaining - State and Nodes

# State schema
class PromptChainState(TypedDict):
    document: str
    extracted_info: str  # Step 1 output
    transformed_data: str  # Step 2 output
    final_summary: str  # Step 3 output

# Node 1: Extract key information
def extract_info(state: PromptChainState):
    """Extract key facts from document"""
    prompt = f"""Extract the key information from this document:

{state['document']}

List only the main facts:"""

    extracted = llm.invoke(prompt).content
    print(f"📤 Extracted: {extracted[:80]}...")
    return {"extracted_info": extracted}

# Node 2: Transform/structure the data
def transform_data(state: PromptChainState):
    """Structure the extracted information"""
    prompt = f"""Structure this information into categories:

{state['extracted_info']}

Organize by: Who, What, When, Where:"""

    transformed = llm.invoke(prompt).content
    print(f"🔄 Transformed: {transformed[:80]}...")
    return {"transformed_data": transformed}

# Node 3: Create final summary
def create_summary(state: PromptChainState):
    """Generate concise summary"""
    prompt = f"""Create a concise 2-sentence summary:

{state['transformed_data']}

Summary:"""

    summary = llm.invoke(prompt).content
    print(f"📝 Summary: {summary[:80]}...")
    return {"final_summary": summary}

In [ ]:
# Prompt Chaining - Graph Construction

# Build graph (simple linear chain)
chain_workflow = StateGraph(PromptChainState)

# Add nodes in sequence
chain_workflow.add_node("extract", extract_info)
chain_workflow.add_node("transform", transform_data)
chain_workflow.add_node("summarize", create_summary)

# Add edges (linear flow)
chain_workflow.add_edge(START, "extract")
chain_workflow.add_edge("extract", "transform")
chain_workflow.add_edge("transform", "summarize")
chain_workflow.add_edge("summarize", END)

# Compile
prompt_chain_agent = chain_workflow.compile()

print("✅ Prompt Chaining agent built!")

In [ ]:
# Prompt Chaining - Test Example

print("=" * 60)
print("Prompt Chaining Pattern Demo")
print("=" * 60)

# Test document
sample_doc = """
Alice Johnson, the CEO of TechCorp, announced a new AI product launch on December 15, 2024.
The product, called SmartAssist, will be available in San Francisco initially.
The launch event will feature live demos and will be open to the public.
"""

result = prompt_chain_agent.invoke({
    "document": sample_doc,
    "extracted_info": "",
    "transformed_data": "",
    "final_summary": ""
})

print("\n" + "=" * 60)
print("PROCESSING PIPELINE")
print("=" * 60)
print(f"\n1. Original Document:\n{result['document']}")
print(f"\n2. Extracted Info:\n{result['extracted_info']}")
print(f"\n3. Transformed Data:\n{result['transformed_data']}")
print(f"\n4. Final Summary:\n{result['final_summary']}")

# 4. Routing Pattern

## What It Does

Routing classifies inputs and directs them to **specialized handlers** based on their type or category. A central router analyzes the input and sends it to the appropriate specialized process, enabling optimized handling for different input types.

In [ ]:
**Flow Diagram:**
Input → Classifier → [Technical Query] → Technical Handler
                  → [Billing Query] → Billing Handler
                  → [General Query] → General Handler

---

## When to Use

✅ **Use Routing when:**

- Diverse input types need different handling

- Specialized processes for each category

- Want to optimize performance per input type

- Building customer service or classification systems

❌ **Don't use when:**

- All inputs handled the same way

- Only one or two input types

- Classification is unreliable

In [ ]:
# Routing Pattern - State and Nodes

# State schema
class RoutingState(TypedDict):
    query: str
    category: str  # technical, billing, or general
    response: str

# Node 1: Classify input
def classify_query(state: RoutingState):
    """Classify the query into a category"""
    prompt = f"""Classify this customer query into ONE category:
- technical (product issues, bugs, features)
- billing (payments, invoices, pricing)
- general (other questions)

Query: {state['query']}

Category (one word):"""

    category = llm.invoke(prompt).content.strip().lower()

    # Ensure valid category
    if category not in ['technical', 'billing', 'general']:
        category = 'general'

    print(f"🏷️  Classified as: {category}")
    return {"category": category}

# Node 2: Handle technical queries
def handle_technical(state: RoutingState):
    """Specialized technical support handler"""
    response = f"[TECHNICAL SUPPORT] I'll help you with: {state['query']}. Let me check our technical documentation..."
    print(f"🔧 Technical handler activated")
    return {"response": response}

# Node 3: Handle billing queries
def handle_billing(state: RoutingState):
    """Specialized billing handler"""
    response = f"[BILLING DEPARTMENT] Regarding: {state['query']}. Let me access your account details..."
    print(f"💰 Billing handler activated")
    return {"response": response}

# Node 4: Handle general queries
def handle_general(state: RoutingState):
    """General purpose handler"""
    response = f"[GENERAL SUPPORT] Thank you for asking: {state['query']}. Here's what I can tell you..."
    print(f"💬 General handler activated")
    return {"response": response}

In [ ]:
# Routing Pattern - Graph Construction

# Router function: direct to appropriate handler
def route_to_handler(state: RoutingState) -> Literal["technical", "billing", "general"]:
    """Route based on category"""
    return state["category"]

# Build graph
routing_workflow = StateGraph(RoutingState)

# Add nodes
routing_workflow.add_node("classify", classify_query)
routing_workflow.add_node("technical", handle_technical)
routing_workflow.add_node("billing", handle_billing)
routing_workflow.add_node("general", handle_general)

# Add edges
routing_workflow.add_edge(START, "classify")
routing_workflow.add_conditional_edges(
    "classify",
    route_to_handler,
    {
        "technical": "technical",
        "billing": "billing",
        "general": "general"
    }
)
# All handlers go to END
routing_workflow.add_edge("technical", END)
routing_workflow.add_edge("billing", END)
routing_workflow.add_edge("general", END)

# Compile
routing_agent = routing_workflow.compile()

print("✅ Routing agent built!")

In [ ]:
# Routing Pattern - Test Examples

print("=" * 60)
print("Routing Pattern Demo")
print("=" * 60)

# Test different query types
queries = [
    "My app keeps crashing when I click the export button",
    "I was charged twice for my subscription this month",
    "What are your business hours?"
]

for i, query in enumerate(queries, 1):
    print(f"\n--- Query {i} ---")
    result = routing_agent.invoke({
        "query": query,
        "category": "",
        "response": ""
    })
    print(f"Query: {result['query']}")
    print(f"Category: {result['category']}")
    print(f"Response: {result['response']}")
    print()

# 5. Reflection Pattern (Summary)

## What It Does

The **Reflection pattern** enables agents to iteratively improve their outputs through self-critique. The agent generates output, critiques it, and refines based on feedback—repeating until quality is satisfactory.

In [ ]:
**Key Concept:**
Generate → Self-Critique → Refine → Self-Critique → ... → Final Output
           ↑___________________________________|

**When to use:** Content generation, code writing, essay composition where quality improvement matters.

---

## Core Implementation

The pattern uses a loop with three key nodes:

1. **Generate** - Create initial output or refinement

2. **Critique** - Agent evaluates its own work

3. **Decide** - Continue refining or finish

---

## Full Implementation

📖 **See complete working code in:** [reflection_reflexion_agent.ipynb](reflection_reflexion_agent.ipynb)

The notebook includes:

- Full state management with iteration tracking

- Self-critique prompting strategies

- Conditional routing based on quality

- Multiple refinement iterations

- Working examples with output

In [ ]:
# Reflection Pattern - Minimal Example
# (See reflection_reflexion_agent.ipynb for full implementation)

# Simple reflection loop concept
draft = "AI is good for healthcare."

# Iteration 1: Self-critique
critique = llm.invoke(f"Critique this draft: {draft}. What's missing?").content
print(f"Critique: {critique[:80]}...")

# Iteration 2: Refine based on critique
refined = llm.invoke(f"Improve: {draft}\nBased on: {critique}").content
print(f"Refined: {refined[:80]}...")

# Key insight: Agent improves its own output through self-reflection

# 6. Reflexion Pattern (Summary)

## What It Does

The **Reflexion pattern** enables agents to **learn from past failures** across multiple tasks. Unlike Reflection (which improves a single output), Reflexion accumulates learnings from previous attempts and applies them to future tasks, creating a self-improving agent over time.

In [ ]:
**Key Concept:**
Task 1 → Attempt → Fail (score: 3/10) → Store learning
Task 2 → Use learning → Attempt → Better (score: 6/10) → Store learning
Task 3 → Use all learnings → Attempt → Success (score: 9/10)

**When to use:** Multi-task scenarios where agent can learn from failures (debugging, optimization, iterative problem-solving).

---

## Core Implementation

The pattern uses episodic memory with four key components:

1. **Attempt** - Generate solution using past learnings

2. **Evaluate** - External assessment (score/feedback)

3. **Learn** - Extract lessons from failure

4. **Store** - Accumulate learnings for future tasks

---

## Full Implementation

📖 **See complete working code in:** [reflection_reflexion_agent.ipynb](reflection_reflexion_agent.ipynb)

The notebook includes:

- Episodic memory with MemorySaver checkpointer

- Cross-task learning accumulation

- External evaluation simulation

- Multiple task examples showing improvement

- Working demonstrations of learning over time

In [ ]:
# Reflexion Pattern - Minimal Example
# (See reflection_reflexion_agent.ipynb for full implementation)

# Learning across multiple tasks
learnings = []

# Task 1: Attempt and learn from failure
attempt_1 = "Use a simple loop"
score_1 = 3  # Failed
learnings.append("Avoid simple loops for complex problems")

# Task 2: Use past learning
attempt_2 = llm.invoke(f"Solve task. Past learnings: {learnings}").content
score_2 = 7  # Better!
learnings.append("Consider edge cases")

# Task 3: Apply all learnings
attempt_3 = llm.invoke(f"Solve task. Past learnings: {learnings}").content
# Score improves over time as agent learns from past attempts

print(f"Learnings accumulated: {len(learnings)}")
# Key insight: Agent gets better at similar tasks by learning from failures

# 7. Tool-Calling Pattern (Summary)

## What It Does

The **Tool-Calling pattern** enables agents to interact with **external APIs, functions, and databases** to gather real-time information or perform actions. The agent decides which tools to call based on the task, executes them, and uses the results to formulate answers.

In [ ]:
**Key Concept:**
Question → Decide Tool → Call API/Function → Process Result → Answer
           ↑_____________________________| (may loop for multiple tools)

**When to use:** Agents need real-time data (weather, stock prices, database queries), need to perform actions (send emails, update records), or require capabilities beyond LLM knowledge.

---

## Core Implementation

The pattern uses tool binding and execution with key components:

1. **Tool Definition** - Specify available tools with schemas

2. **Tool Selection** - LLM chooses appropriate tool(s)

3. **Tool Execution** - Call external function/API

4. **Result Integration** - Incorporate results into response

---

## Full Implementation

📖 **See complete working code in:** [agent_with_tools.ipynb](agent_with_tools.ipynb)

The notebook includes:

- Tool binding with LangChain

- Function calling with OpenAI models

- Multiple tool examples (search, calculator, database)

- Tool result processing

- Error handling for tool failures

- Working examples with real tool executions

In [ ]:
# Tool-Calling Pattern - Minimal Example
# (See agent_with_tools.ipynb for full implementation)

# Define a simple tool
def get_weather(city: str) -> str:
    """Simulated weather API"""
    return f"Weather in {city}: Sunny, 72°F"

# Agent decides to use tool based on question
question = "What's the weather in San Francisco?"

# LLM determines it needs weather tool
tool_choice = "get_weather"  # In real implementation, LLM chooses this

# Execute tool
if tool_choice == "get_weather":
    result = get_weather("San Francisco")
    answer = llm.invoke(f"Based on: {result}, answer: {question}").content
    print(f"Answer: {answer}")

# Key insight: Agent extends capabilities by calling external functions/APIs

---

# Additional Agentic Patterns

Beyond the 7 core patterns above, here are other important patterns worth exploring:

## Advanced Patterns Reference Table

| **Pattern** | **Description** | **Use Case** | **Reference** |

|-------------|----------------|--------------|---------------|

| **Human-in-the-Loop** | Pauses execution for human approval/input at critical decision points | High-stakes decisions, content moderation, approval workflows | [human_in_the_loop.ipynb](human_in_the_loop.ipynb) |

| **Memory Patterns** | Agents maintain conversation history and context across interactions | Chatbots, customer service, personalized assistants | [agent_memory.ipynb](agent_memory.ipynb) |

| **Session Management** | Persistent state across multiple agent runs with checkpointing | Production deployments, multi-session workflows | [checkpoint_session_management.ipynb](checkpoint_session_management.ipynb) |

| **Orchestrator-Worker** | Central orchestrator delegates specialized tasks to worker agents | Multi-agent systems, complex pipelines, parallel processing | Complex multi-agent coordination |

| **Tree of Thoughts** | Explores multiple reasoning paths simultaneously, then selects best | Complex problem-solving, creative tasks, strategic planning | Advanced reasoning tasks |

---

## When Each Pattern Excels

**Quick Selection Guide:**

- **Need external tools/APIs?** → ReAct or Tool-Calling

- **Complex multi-step task?** → Plan-and-Execute

- **Sequential processing pipeline?** → Prompt Chaining

- **Different input types need different handling?** → Routing

- **Improve single output quality?** → Reflection

- **Learn from failures across tasks?** → Reflexion

- **Human oversight required?** → Human-in-the-Loop

- **Need conversation context?** → Memory Patterns

- **Production deployment?** → Session Management

- **Multiple specialized agents?** → Orchestrator-Worker

- **Explore multiple solutions?** → Tree of Thoughts

---

# Pattern Selection Decision Tree

## How to Choose the Right Pattern

Follow this decision tree to select the appropriate pattern for your task:

```
START: What is your primary goal?
│
├─ Need to USE EXTERNAL TOOLS/APIs?
│  └─ YES → Use **Tool-Calling** or **ReAct**
│     ├─ Simple tool use → Tool-Calling
│     └─ Need reasoning about tool use → ReAct
│
├─ Need to ROUTE different INPUT TYPES?
│  └─ YES → Use **Routing**
│     └─ Classify input → Send to specialized handlers
│
├─ Need to BREAK DOWN complex multi-step task?
│  └─ YES → Use **Plan-and-Execute**
│     └─ Create plan → Execute steps sequentially
│
├─ Need to PROCESS through sequential stages?
│  └─ YES → Use **Prompt Chaining**
│     └─ Stage 1 → Stage 2 → Stage 3 → Output
│
├─ Need to IMPROVE output quality?
│  └─ YES → Use **Reflection**
│     └─ Generate → Critique → Refine → Repeat
│
├─ Need to LEARN from past failures?
│  └─ YES → Use **Reflexion**
│     └─ Attempt → Fail → Learn → Retry with knowledge
│
└─ Need HUMAN APPROVAL at key points?
   └─ YES → Use **Human-in-the-Loop**
      └─ Execute → Pause for approval → Continue
```

---

## Combining Patterns

Patterns can be combined for powerful agent systems:

### Common Combinations:

1. **ReAct + Reflection**

   - Use ReAct for tool-based reasoning

   - Add Reflection to improve tool selection quality

   - Example: Research agent that reflects on search query quality

2. **Routing + Plan-and-Execute**

   - Route inputs to different planning strategies

   - Each route has its own plan-execute pipeline

   - Example: Customer service system with specialized workflows

3. **Plan-and-Execute + Tool-Calling**

   - Create plan with multiple steps

   - Each step can call appropriate tools

   - Example: Data analysis pipeline with API calls

4. **Reflection + Memory**

   - Agent improves outputs over time

   - Stores successful patterns in memory

   - Example: Writing assistant that learns user preferences

5. **Routing + Human-in-the-Loop**

   - Route high-stakes decisions to human approval

   - Auto-handle routine cases

   - Example: Content moderation system

---

# Best Practices & Summary

## Implementation Best Practices

### 1. Start Simple

- Begin with the simplest pattern that solves your problem

- Add complexity only when needed

- Test each pattern independently before combining

### 2. State Management

- Keep state schemas clear and minimal

- Use `TypedDict` for type safety

- Use `Annotated[list, add]` for accumulating results

- Document what each state field represents

### 3. Node Design

- Each node should do ONE thing well

- Keep nodes pure (deterministic outputs for given inputs)

- Add logging/printing for debugging

- Handle errors gracefully in production

### 4. Graph Construction

- Use clear node names that describe their purpose

- Prefer conditional edges for dynamic routing

- Always compile with checkpointer for stateful agents

- Test graph structure before adding complex logic

### 5. Testing

- Test with simple inputs first

- Verify state transitions work correctly

- Check edge cases (empty inputs, long inputs, etc.)

- Monitor LLM token usage and costs

### 6. Production Considerations

- Add error handling and retries

- Implement timeouts for LLM calls

- Use checkpointers for session persistence

- Monitor and log agent decisions

- Add Human-in-the-Loop for high-stakes decisions

---

## Pattern Summary Cheat Sheet

| **When you need...** | **Use this pattern** | **Key benefit** |

|---------------------|---------------------|-----------------|

| External data/tools | Tool-Calling or ReAct | Extends agent capabilities |

| Multi-step problem | Plan-and-Execute | Structured reasoning |

| Sequential processing | Prompt Chaining | Clear pipeline stages |

| Different input types | Routing | Specialized handling |

| Better output quality | Reflection | Self-improvement |

| Learning over time | Reflexion | Gets smarter with experience |

| Human oversight | Human-in-the-Loop | Safety and control |

---

## Next Steps

### Explore Full Implementations:

- 📖 [reflection_reflexion_agent.ipynb](reflection_reflexion_agent.ipynb) - Deep dive on Reflection/Reflexion with full examples

- 📖 [agent_with_tools.ipynb](agent_with_tools.ipynb) - Complete tool integration patterns

- 📖 [human_in_the_loop.ipynb](human_in_the_loop.ipynb) - Interrupt and approval workflows

- 📖 [agent_memory.ipynb](agent_memory.ipynb) - Memory architecture patterns

- 📖 [checkpoint_session_management.ipynb](checkpoint_session_management.ipynb) - Production deployment patterns

### Learn More:

- **LangGraph Documentation**: [https://langchain-ai.github.io/langgraph/](https://langchain-ai.github.io/langgraph/)

- **LangChain Patterns**: [https://python.langchain.com/docs/](https://python.langchain.com/docs/)

- **Agent Design Principles**: Research papers on ReAct, Reflexion, and Tree of Thoughts

---

## Conclusion

Agentic design patterns provide proven solutions for building intelligent agents. By understanding when and how to use each pattern, you can:

✅ Build more reliable and maintainable agents

✅ Solve complex problems systematically

✅ Combine patterns for powerful capabilities

✅ Deploy production-ready agent systems

**Remember**: Start with the simplest pattern that works, test thoroughly, and iterate based on real-world performance.

Happy building! 🚀